# Phase 0 spike: SQL Server egress test

Purpose: confirm this Databricks compute can actually reach the public SQL Server host (`xomdata_dataset.web_analytics`) before building the real `dlt` ingestion pipeline around that assumption.

**This notebook must run on Databricks compute** (attach it to a cluster / run it as a notebook task), not locally — local-machine connectivity was already confirmed via `dbt debug`, that's not what's in question here.

As a bonus, once connected this also pulls `INFORMATION_SCHEMA.COLUMNS` for the source tables, which resolves the other open item: confirming a rising-ID or `created_at` column exists per table for `dlt` incremental cursors later.

**Do not save real credentials into this file.** Fill in the widgets at run time; clear them before committing.

In [0]:
# %pip install pymssql

In [0]:
# dbutils.library.restartPython()

In [0]:
# Connection parameters as widgets so nothing sensitive lives in the notebook source.
# Fill these in from the run UI each time; they are not persisted to the file.
dbutils.widgets.text("host", "", "SQL Server host")
dbutils.widgets.text("port", "1433", "Port")
dbutils.widgets.text("database", "xomdata_dataset", "Database")
dbutils.widgets.text("schema", "web_analytics", "Schema")
dbutils.widgets.text("user", "", "User")
dbutils.widgets.text("password", "", "Password")

In [0]:
import time

host = dbutils.widgets.get("host")
port = int(dbutils.widgets.get("port"))
database = dbutils.widgets.get("database")
schema = dbutils.widgets.get("schema")
user = dbutils.widgets.get("user")
password = dbutils.widgets.get("password")

assert all([host, user, password]), "Fill in host, user, and password widgets before running."

jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName={database};encrypt=true;trustServerCertificate=true;loginTimeout=15"

print(f"Attempting connection to {host}:{port}/{database} ...")
start = time.monotonic()
try:
    df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("user", user) \
        .option("password", password) \
        .option("query", "SELECT 1 AS test") \
        .load()
    df.show()
    elapsed = time.monotonic() - start
    print(f"CONNECTED in {elapsed:.2f}s -- egress is open, Databricks can reach this SQL Server.")
except Exception as e:
    elapsed = time.monotonic() - start
    print(f"FAILED after {elapsed:.2f}s -- egress is likely blocked (or credentials/host are wrong).")
    print(f"Error: {type(e).__name__}: {e}")
    raise

In [0]:
# Sanity-check the connection works for real queries, not just the TCP handshake.
df_sample = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", user) \
    .option("password", password) \
    .option("dbtable", f"{schema}.website_sessions") \
    .load() \
    .limit(3)

df_sample.show(truncate=False)

In [0]:
# Bonus: real column-level schema for the source tables, to pick incremental cursor
# columns (rising ID / created_at) for each dlt resource later.
df_schema = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", user) \
    .option("password", password) \
    .option("query", f"""
        SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE, IS_NULLABLE, ORDINAL_POSITION
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{schema}'
    """) \
    .load() \
    .orderBy("TABLE_NAME", "ORDINAL_POSITION")

df_schema.show(1000, truncate=False)

In [0]:
print("Done. Clear the password widget before leaving this notebook.")